In [ ]:
import os
import base64
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

endpoint = os.getenv("OPENAI_ENDPOINT")
api_key = os.getenv("ANTHROPIC_API_KEY")
chat_model = os.getenv("OPENAI_MODEL")

client = OpenAI(
    api_key=api_key,
    base_url=endpoint,
)

### Convert an image to Base64 encoded

In [ ]:
import base64
from mimetypes import guess_type

# Function to encode a local image into data URL 
def local_image_to_data_url(image_path):
    # Guess the MIME type of the image based on the file extension
    mime_type, _ = guess_type(image_path)
    if mime_type is None:
        mime_type = 'application/octet-stream'  # Default MIME type if none is found

    # Read and encode the image file
    with open(image_path, "rb") as image_file:
        base64_encoded_data = base64.b64encode(image_file.read()).decode('utf-8')

    # Construct the data URL
    return base64_encoded_data

In [ ]:
base64_sample = local_image_to_data_url("./data/sample01_resized.png")
base64_check = local_image_to_data_url("./data/check01_resized.png")
url_sample = {"url": "data:image/jpeg;base64," + base64_sample}
url_check = {"url": "data:image/jpeg;base64," + base64_check}

### Responses

In [ ]:
from pathlib import Path
user_prompt_template = Path("./user_prompt_template_01.txt").read_text(encoding="utf-8")
user_prompt_skill = Path("./.claude/skills/missing-dimension-check/SKILL.md").read_text(encoding="utf-8")

response_image_detail = "high"         # "low" first; use "high" if OCR quality is insufficient

response_input = [
    {
        "role": "user",
        "content": [
            {"type": "input_text", "text": user_prompt_template},
            {"type": "input_text", "text": "Attachment 1"},
            {"type": "input_image", "image_url": url_sample["url"], "detail": response_image_detail},
            {"type": "input_text", "text": "Attachment 2"},
            {"type": "input_image", "image_url": url_check["url"], "detail": response_image_detail},
            {"type": "input_text", "text":"Finally, mark the missing dimensions with red circles on check01.png, and be sure to return a link to the generated image file."},
            {"type": "input_text", "text": user_prompt_skill}
        ],
    }
]

In [ ]:
response_reasoning_effort = "high"  # "low", "medium", "high"
#response_max_output_tokens = 4000
response_timeout_seconds = 600

stream = client.responses.create(
    model=chat_model,
    input=response_input,
    reasoning={"effort": response_reasoning_effort},
    #max_output_tokens=response_max_output_tokens,
    timeout=response_timeout_seconds,
    stream=True,
    tools=[
        {
            "type": "code_interpreter",
            "container": {"type": "auto", "memory_limit": "4g"}
        }
    ],
)

answer_parts = []
response = None

for event in stream:
    event_type = getattr(event, "type", None)

    if event_type == "response.output_text.delta":
        delta = getattr(event, "delta", "")
        print(delta, end="", flush=True)
        answer_parts.append(delta)
    elif event_type == "response.completed":
        response = getattr(event, "response", None)
    elif event_type == "response.failed":
        error = getattr(event, "error", None)
        raise RuntimeError(f"Response failed: {error}")

print()
answer = "".join(answer_parts)


### Retrieve the created image

In [ ]:
from pathlib import Path

out_dir = Path("out")
out_dir.mkdir(exist_ok=True)

downloaded = []

for item in response.output:
    if getattr(item, "type", None) != "message":
        continue

    for content in item.content:
        if getattr(content, "type", None) != "output_text":
            continue

        for ann in getattr(content, "annotations", []) or []:
            if getattr(ann, "type", None) != "container_file_citation":
                continue

            filename = Path(ann.filename).name
            out_path = out_dir / filename

            file_content = client.containers.files.content.retrieve(
                container_id=ann.container_id,
                file_id=ann.file_id,
            )
            file_content.write_to_file(out_path)

            downloaded.append(out_path)
            print(f"Downloaded: {filename} -> {out_path}")

downloaded

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(downloaded[0])))